# 2. Methodology

## Feature Engineering and Preprocessing

In [1]:
%pip install imbalanced-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\User\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [2]:
import sys
import os
import pandas as pd
# Adds the parent directory to the search path
sys.path.append(os.path.abspath(".."))

In [3]:
from src.preprocessing import (
    split_features_target,
    stratified_train_test_split,
    scale_features,
    apply_smote,
)
from src.config import RAW_DATA_PATH, RANDOM_SEED, PROCESSED_DIR
from src.data_loader import load_raw_data
from src.logging_utils import get_logger

df = load_raw_data(RAW_DATA_PATH)
logger = get_logger("notebook_02")

X, y = split_features_target(df, target_col="Outcome")
X_train, X_test, y_train, y_test = stratified_train_test_split(X, y)

X_train_scaled, X_test_scaled, scaler = scale_features(X_train, X_test)
X_train_balanced, y_train_balanced = apply_smote(X_train_scaled, y_train)

# Save processed data
X_train_scaled.to_csv(PROCESSED_DIR / "X_train_scaled.csv", index=True)
X_test_scaled.to_csv(PROCESSED_DIR / "X_test_scaled.csv", index=True)
y_train.to_csv(PROCESSED_DIR / "y_train.csv", index=True)
y_test.to_csv(PROCESSED_DIR / "y_test.csv", index=True)
X_train_balanced.to_csv(PROCESSED_DIR / "X_train_balanced.csv", index=False)
y_train_balanced.to_csv(PROCESSED_DIR / "y_train_balanced.csv", index=False)

[2026-03-25 11:02:10] [src.data_loader] [INFO] Loading raw data from D:\Portfolio Projects\diabetes-ml-prediction\data\raw\diabetes.csv
[2026-03-25 11:02:10] [src.data_loader] [INFO] Loaded dataset with shape (768, 9)
[2026-03-25 11:02:10] [src.preprocessing] [INFO] Splitting features and target
[2026-03-25 11:02:10] [src.preprocessing] [INFO] Features shape: (768, 8), Target shape: (768,)
[2026-03-25 11:02:10] [src.preprocessing] [INFO] Performing stratified train-test split
[2026-03-25 11:02:10] [src.preprocessing] [INFO] Train size: 576, Test size: 192
[2026-03-25 11:02:10] [src.preprocessing] [INFO] Scaling features with StandardScaler
[2026-03-25 11:02:10] [src.preprocessing] [INFO] Feature scaling complete
[2026-03-25 11:02:10] [src.preprocessing] [INFO] Applying SMOTE to balance classes
[2026-03-25 11:02:10] [src.preprocessing] [INFO] After SMOTE - X_balanced: (750, 8), y_balanced: (750,)


## Model Selection and Hyperparameter Tuning

In [4]:
from src.modeling import (
    get_base_models,
    tune_gradient_boosting,
    tune_random_forest,
    tune_xgboost,
    cross_validate_models,
)

base_models = get_base_models()

gb_best = tune_gradient_boosting(X_train_balanced, y_train_balanced)
rf_best = tune_random_forest(X_train_balanced, y_train_balanced)
xgb_best = tune_xgboost(X_train_balanced, y_train_balanced)

# Replace base models with tuned versions
base_models["Gradient Boosting"] = gb_best
base_models["Random Forest"] = rf_best
base_models["XGBoost"] = xgb_best

cv_results = cross_validate_models(base_models, X_train_balanced, y_train_balanced)
cv_results

[2026-03-25 11:02:10] [src.modeling] [INFO] Initializing base models
[2026-03-25 11:02:10] [src.modeling] [INFO] Tuning Gradient Boosting hyperparameters
[2026-03-25 11:11:03] [src.modeling] [INFO] Best GB params: {'learning_rate': 0.1, 'max_depth': 6, 'min_samples_split': 5, 'n_estimators': 200, 'subsample': 0.8}, ROC-AUC: 0.8921
[2026-03-25 11:11:03] [src.modeling] [INFO] Tuning Random Forest hyperparameters
[2026-03-25 11:14:30] [src.modeling] [INFO] Best RF params: {'max_depth': 15, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}, ROC-AUC: 0.8929
[2026-03-25 11:14:30] [src.modeling] [INFO] Tuning XGBoost hyperparameters
[2026-03-25 11:16:26] [src.modeling] [INFO] Best XGB params: {'colsample_bytree': 1.0, 'learning_rate': 0.05, 'max_depth': 6, 'n_estimators': 300, 'subsample': 0.8}, ROC-AUC: 0.8783
[2026-03-25 11:16:26] [src.modeling] [INFO] Running cross-validation for all models
[2026-03-25 11:16:26] [src.modeling] [INFO] Logistic Regre

,Model,CV Accuracy Mean,CV Accuracy Std,CV ROC-AUC Mean,CV ROC-AUC Std
0,Logistic Regression,0.734667,0.018571,0.822791,0.007630
1,Random Forest,0.816000,0.044141,0.884196,0.024700
2,Gradient Boosting,0.806667,0.016330,0.883449,0.017325
3,SVM (RBF),0.746667,0.020221,0.844942,0.009754
4,XGBoost,0.802667,0.029695,0.874524,0.022612


## Individual Model Test Set Performance

In [5]:
from src.evaluation import evaluate_model, get_confusion_and_report, compute_roc_pr_curves
from src.visualization import plot_roc_curves

results = []
roc_curves = {}

for name, model in base_models.items():
    model.fit(X_train_balanced, y_train_balanced)
    metrics = evaluate_model(model, X_test_scaled, y_test, model_name=name)
    results.append(metrics)

    fpr, tpr, precision, recall = compute_roc_pr_curves(model, X_test_scaled, y_test)
    roc_curves[name] = {"fpr": fpr, "tpr": tpr}

results_df = pd.DataFrame(results)
results_df

[2026-03-25 11:16:51] [src.evaluation] [INFO] Evaluating model: Logistic Regression
[2026-03-25 11:16:51] [src.evaluation] [INFO] Logistic Regression - Acc: 0.7969, Prec: 0.6944, Rec: 0.7463, F1: 0.7194, ROC-AUC: 0.8683
[2026-03-25 11:16:51] [src.evaluation] [INFO] Evaluating model: Random Forest
[2026-03-25 11:16:51] [src.evaluation] [INFO] Random Forest - Acc: 0.7812, Prec: 0.6582, Rec: 0.7761, F1: 0.7123, ROC-AUC: 0.8542
[2026-03-25 11:16:53] [src.evaluation] [INFO] Evaluating model: Gradient Boosting
[2026-03-25 11:16:53] [src.evaluation] [INFO] Gradient Boosting - Acc: 0.7656, Prec: 0.6447, Rec: 0.7313, F1: 0.6853, ROC-AUC: 0.8239
[2026-03-25 11:16:53] [src.evaluation] [INFO] Evaluating model: SVM (RBF)
[2026-03-25 11:16:53] [src.evaluation] [INFO] SVM (RBF) - Acc: 0.7500, Prec: 0.6234, Rec: 0.7164, F1: 0.6667, ROC-AUC: 0.8576
[2026-03-25 11:16:53] [src.evaluation] [INFO] Evaluating model: XGBoost
[2026-03-25 11:16:53] [src.evaluation] [INFO] XGBoost - Acc: 0.7656, Prec: 0.6375, R

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Logistic Regression,0.796875,0.694444,0.746269,0.719424,0.868299
1,Random Forest,0.781250,0.658228,0.776119,0.712329,0.854209
2,Gradient Boosting,0.765625,0.644737,0.731343,0.685315,0.823881
3,SVM (RBF),0.750000,0.623377,0.716418,0.666667,0.857552
4,XGBoost,0.765625,0.637500,0.761194,0.693878,0.825910


In [7]:
from pathlib import Path
from src.config import FIGURES_DIR

plot_roc_curves(roc_curves, FIGURES_DIR / "roc_curves.png")

[2026-03-25 11:17:08] [src.visualization] [INFO] Plotting ROC curves
[2026-03-25 11:17:09] [src.visualization] [INFO] Saved ROC curves figure to D:\Portfolio Projects\diabetes-ml-prediction\reports\figures\roc_curves.png


In [8]:
logreg_model = base_models["Logistic Regression"]
cm, report = get_confusion_and_report(
    logreg_model, X_test_scaled, y_test, target_names=["No Diabetes", "Diabetes"]
)
print("Confusion Matrix:\n", cm)
print("\nClassification Report:\n", report)

Confusion Matrix:
 [[103  22]
 [ 17  50]]

Classification Report:
               precision    recall  f1-score   support

 No Diabetes       0.86      0.82      0.84       125
    Diabetes       0.69      0.75      0.72        67

    accuracy                           0.80       192
   macro avg       0.78      0.79      0.78       192
weighted avg       0.80      0.80      0.80       192

